# Lesson 11A: Self-Organizing Maps (SOM) - Theory

Learn neural network-based clustering and visualization.

**Self-Organizing Map (SOM)** is an unsupervised neural network that:
- Projects high-dimensional data to 2D grid
- Preserves topological structure
- Uses competitive learning

**Algorithm:**
1. Initialize random weight vectors for each neuron
2. For each input sample:
   - Find Best Matching Unit (BMU) - closest neuron
   - Update BMU and neighbors toward input
3. Decrease learning rate and neighborhood over time

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

## SOM Update Rule

For winning neuron c and input x:

**w_i(t+1) = w_i(t) + η(t) · h_ci(t) · [x(t) - w_i(t)]**

Where:
- η(t) = learning rate (decreases over time)
- h_ci(t) = neighborhood function (Gaussian around BMU)
- w_i = weight vector of neuron i

In [ ]:
# Simple SOM implementation
class SimpleSOM:
    def __init__(self, grid_size=(10, 10), input_dim=2, learning_rate=0.5):
        self.grid_size = grid_size
        self.input_dim = input_dim
        self.lr = learning_rate
        # Initialize random weights
        self.weights = np.random.randn(grid_size[0], grid_size[1], input_dim)
    
    def find_bmu(self, x):
        """Find Best Matching Unit"""
        distances = np.linalg.norm(self.weights - x, axis=2)
        return np.unravel_index(np.argmin(distances), self.grid_size)
    
    def update_weights(self, x, bmu, iteration, max_iter):
        """Update weights using Gaussian neighborhood"""
        lr = self.lr * (1 - iteration / max_iter)
        radius = max(self.grid_size) / 2 * (1 - iteration / max_iter)
        
        for i in range(self.grid_size[0]):
            for j in range(self.grid_size[1]):
                dist_to_bmu = np.sqrt((i - bmu[0])**2 + (j - bmu[1])**2)
                if dist_to_bmu <= radius:
                    influence = np.exp(-dist_to_bmu**2 / (2 * radius**2))
                    self.weights[i, j] += lr * influence * (x - self.weights[i, j])

# Test on simple data
from sklearn.datasets import make_blobs
X, _ = make_blobs(n_samples=200, centers=3, n_features=2, random_state=42)

som = SimpleSOM(grid_size=(8, 8), input_dim=2)

# Train
for iteration in range(100):
    x = X[np.random.randint(0, len(X))]
    bmu = som.find_bmu(x)
    som.update_weights(x, bmu, iteration, 100)

# Visualize
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], alpha=0.5)
plt.title('Original Data')
plt.subplot(1, 2, 2)
weights_2d = som.weights.reshape(-1, 2)
plt.scatter(weights_2d[:, 0], weights_2d[:, 1], c='red', marker='s')
plt.title('SOM Neurons')
plt.tight_layout()
plt.show()
print("✅ Simple SOM trained!")